In [ ]:
# Project is installed as editable (pip install -e .)
# so src is importable directly -- no sys.path manipulation needed.
# If you see ModuleNotFoundError, make sure Jupyter is using the venv kernel:
#   venv/Scripts/python -m ipykernel install --user --name=wind-resource-gis
print("Kernel ready.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.config import DATA_PROCESSED

stations = pd.read_parquet(DATA_PROCESSED / "stations_summary.parquet")
grid = pd.read_parquet(DATA_PROCESSED / "grid_summary.parquet")
print(f"Stations: {len(stations)} | Grid: {len(grid)}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(grid["lon"], grid["lat"], c=grid["ws10_mean"],
           cmap="viridis", s=40, label="Grid points")
ax.scatter(stations["lon"], stations["lat"], c="red", s=80,
           marker="^", label="NOAA stations", edgecolor="black")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Wind data coverage -- Northeast US")
ax.legend()
plt.colorbar(ax.collections[0], label="Mean wind speed at 10m (m/s)")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(grid["ws10_mean"], bins=20, edgecolor="black")
axes[0].set_xlabel("Mean wind speed at 10m (m/s)")
axes[0].set_ylabel("Grid points")
axes[0].set_title("10m wind speed distribution (ERA5 grid)")

axes[1].hist(grid["ws100_mean"], bins=20, edgecolor="black", color="orange")
axes[1].set_xlabel("Mean wind speed at 100m (m/s)")
axes[1].set_title("100m wind speed distribution")
plt.tight_layout(); plt.show()

In [ ]:
merged = stations.merge(
    grid.rename(columns={"ws10_mean": "ws10_grid"}),
    on=["lat", "lon"], how="inner",
)
print("Stations match data? Won't match exactly -- Phase 2 interpolates.")
print("
Quick comparison stats:")
print(f"Station mean wind: {stations["ws10_mean"].mean():.2f} m/s")
print(f"Grid mean wind:    {grid["ws10_mean"].mean():.2f} m/s")